In [ ]:
# EXP 3: Logistic Regression & Log Loss Evaluation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
Y = iris.target

print("Dataset shape:", X.shape)
print("Target classes:", np.unique(Y))

Xtr, Xte, Ytr, Yte = train_test_split(
    X, Y, test_size=0.3, random_state=42, stratify=Y)
print("Training samples:", len(Xtr))
print("Testing samples:", len(Xte))

s = StandardScaler()
Xtr, Xte = s.fit_transform(Xtr), s.transform(Xte)

m = LogisticRegression(max_iter=5000, solver="lbfgs")
m.fit(Xtr, Ytr)

print("Training Accuracy:", accuracy_score(Ytr, m.predict(Xtr)))
print("Testing Accuracy:", accuracy_score(Yte, m.predict(Xte)))

train_loss, test_loss = [], []
iterations = range(100, 1001, 100)

for i in iterations:
    model = LogisticRegression(max_iter=i, solver="lbfgs").fit(Xtr, Ytr)
    train_loss.append(log_loss(Ytr, model.predict_proba(Xtr)))
    test_loss.append(log_loss(Yte, model.predict_proba(Xte)))

plt.figure(figsize=(7, 5))
plt.plot(iterations, train_loss, marker="o", label="Training Loss")
plt.plot(iterations, test_loss, marker="s", label="Testing Loss")
plt.xlabel("Maximum Iterations")
plt.ylabel("Log Loss")
plt.title("Logistic Regression Loss Function")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# EXP 4: Backpropagation
import math

x, t = 1, 0.5
w0, bh0, v0, bo0 = 0.5, 0, 0.7, 0
alpha, epochs = 0.5, 5

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

def sigmoid_derivative(y):
    return y * (1 - y)

def tanh(z):
    return math.tanh(z)

def tanh_derivative(y):
    return 1 - y * y

def relu(z):
    return max(0, z)

def relu_derivative(z):
    return 1 if z > 0 else 0

act = {"Sigmoid": sigmoid, "Tanh": tanh, "ReLU": relu}
deriv = {"Sigmoid": sigmoid_derivative,
         "Tanh": tanh_derivative, "ReLU": relu_derivative}

def train(name):
    w, bh, v, bo = w0, bh0, v0, bo0
    print(name)
    print("Epoch\t x\t t\t w\t bh\t v\t bo\t Error")

    for epoch in range(1, epochs + 1):
        Hin = x * w + bh
        H = act[name](Hin)

        Yin = H * v + bo
        Y = act[name](Yin)

        error = 0.5 * (t - Y) ** 2

        delta_o = (Y - t) * deriv[name](
            Yin if name == "ReLU" else Y)

        delta_h = (v * delta_o) * deriv[name](
            Hin if name == "ReLU" else H)

        v -= alpha * delta_o * H
        bo -= alpha * delta_o
        w -= alpha * delta_h * x
        bh -= alpha * delta_h

        print(epoch, x, t, round(w, 6), round(bh, 6),
              round(v, 6), round(bo, 6), round(error, 8),
              sep="\t")

train("Sigmoid")
train("Tanh")
train("ReLU")

In [ ]:
# EXP 5: Multi-Layer Perceptron
import numpy as np

X = np.array([[0.05], [0.10]])
T = np.array([[0.01], [0.99]])

W = np.array([[0.15, 0.20], [0.25, 0.30]])
bh = np.array([[0.35], [0.35]])
V = np.array([[0.40, 0.45], [0.50, 0.55]])
bo = np.array([[0.60], [0.60]])

alpha, epochs = 0.5, 2

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

for epoch in range(1, epochs + 1):
    Hin = W.T @ X + bh
    H = sigmoid(Hin)
    Yin = V.T @ H + bo
    Y = sigmoid(Yin)

    Error = 0.5 * np.sum((T - Y) ** 2)
    delta_o = (Y - T) * Y * (1 - Y)
    delta_h = (V @ delta_o) * H * (1 - H)

    V -= alpha * (H @ delta_o.T)
    bo -= alpha * delta_o
    W -= alpha * (X @ delta_h.T)
    bh -= alpha * delta_h


    print(f"\nEpoch {epoch}")
    print("Hin:", Hin.ravel().round(4))
    print("H:", H.ravel().round(4))
    print("Yin:", Yin.ravel().round(4))
    print("Y:", Y.ravel().round(4))
    print(f"Error = {Error:.4f}")
    print("Output delta:", delta_o.ravel().round(4))
    print("Hidden delta:", delta_h.ravel().round(4))
    print("W:\n", W.round(4))
    print("bh:", bh.ravel().round(4))
    print("V:\n", V.round(4))
    print("bo:", bo.ravel().round(4))

In [ ]:
# EXP 6: Batch, Stochastic & Mini-Batch Gradient Descent
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler

# Load and preprocess data
X, target = load_wine(return_X_y=True)
X = StandardScaler().fit_transform(X)
T = np.eye(3)[target]

# Activation and loss functions
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

def mse_loss(Y, T):
    return np.mean((Y - T) ** 2)

# Parameters
epochs, lr, hidden, batch_size = 100, 0.1, 5, 16

# Training function
def train(X, T, method):
    W = np.random.randn(13, hidden) * 0.1
    V = np.random.randn(hidden, 3) * 0.1
    bh, bo = np.zeros((hidden, 1)), np.zeros((3, 1))
    losses = []

    for epoch in range(epochs):
        if method == "Batch GD":
            Xb, Tb = X, T
        elif method == "SGD":
            i = np.random.randint(len(X))
            Xb, Tb = X[i:i+1], T[i:i+1]
        else:
            idx = np.random.choice(len(X), batch_size, replace=False)
            Xb, Tb = X[idx], T[idx]

        Xb, Tb = Xb.T, Tb.T
        Hin = W.T @ Xb + bh
        H = sigmoid(Hin)
        Y = sigmoid(V.T @ H + bo)
        error = Y - Tb

        delta_o = error * sigmoid_derivative(V.T @ H + bo)
        delta_h = (V @ delta_o) * sigmoid_derivative(Hin)
        n = Xb.shape[1]

        dV = H @ delta_o.T / n
        dbo = np.mean(delta_o, axis=1, keepdims=True)
        dW = Xb @ delta_h.T / n
        dbh = np.mean(delta_h, axis=1, keepdims=True)

        V -= lr * dV
        bo -= lr * dbo
        W -= lr * dW
        bh -= lr * dbh

        # Calculate loss using the entire dataset
        Xa, Ta = X.T, T.T
        Ha = sigmoid(W.T @ Xa + bh)
        Ya = sigmoid(V.T @ Ha + bo)
        losses.append(mse_loss(Ya, Ta))

    return losses

# Train all three methods
np.random.seed(42)
batch_loss = train(X, T, "Batch GD")
np.random.seed(42)
sgd_loss = train(X, T, "SGD")
np.random.seed(42)
mini_loss = train(X, T, "Mini-Batch GD")

# Plot losses
plt.figure(figsize=(8, 5))
for loss, label in [(batch_loss, "Batch GD"),
                    (sgd_loss, "SGD"),
                    (mini_loss, "Mini-Batch GD")]:
    plt.plot(loss, label=label)

plt.xlabel("Epochs")
plt.ylabel("MSE Loss")
plt.title("Wine Dataset - MLP with Sigmoid Activation")
plt.legend()
plt.grid(True)
plt.show()

# Final MSE
print("\nFinal MSE Loss")
print("Batch GD:", round(batch_loss[-1], 4))
print("SGD:", round(sgd_loss[-1], 4))
print("Mini-Batch GD:", round(mini_loss[-1], 4))

In [ ]:
# EXP 7: Weight Inizialization Techniques
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Load and preprocess MNIST
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()
X_train = X_train.reshape(-1, 784).astype("float32") / 255.0
X_test = X_test.reshape(-1, 784).astype("float32") / 255.0

# Parameters
input_size, hidden_size, output_size = 784, 128, 10
epochs, batch_size, learning_rate = 30, 128, 0.001

# Initialization methods
experiments = {
    "Zero": (tf.keras.initializers.Zeros(), "sigmoid"),
    "Random": (tf.keras.initializers.RandomNormal(stddev=0.01), "sigmoid"),
    "XavierNormal": (tf.keras.initializers.GlorotNormal(), "sigmoid"),
    "XavierUniform": (tf.keras.initializers.GlorotUniform(), "tanh"),
    "He-relu": (tf.keras.initializers.HeNormal(), "relu"),
    "He+leaky relu": (tf.keras.initializers.HeNormal(), "leaky_relu")
}

results, accuracy_history, loss_history = {}, {}, {}

for name, (initializer, activation) in experiments.items():
    print("Running:", name)
    tf.keras.utils.set_random_seed(42)

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_size,)),
        tf.keras.layers.Dense(hidden_size, kernel_initializer=initializer)
    ])

    if activation == "leaky_relu":
        model.add(tf.keras.layers.LeakyReLU(negative_slope=0.01))
    else:
        model.layers[-1].activation = tf.keras.activations.get(activation)

    model.add(tf.keras.layers.Dense(output_size, activation="softmax"))

    model.compile(
        optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    history = model.fit(
        X_train, y_train, epochs=epochs,
        batch_size=batch_size, verbose=0
    )

    train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    results[name] = (train_acc, test_acc, train_loss, test_loss)
    accuracy_history[name] = history.history["accuracy"]
    loss_history[name] = history.history["loss"]

    print(f"Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")
    print(f"Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")

# Display results
print("\nMethod\t\tTrain Acc\tTest Acc")
for name, values in results.items():
    print(f"{name}\t{values[0]:.4f}\t\t{values[1]:.4f}")

print("\nMethod\t\tTrain Loss\tTest Loss")
for name, values in results.items():
    print(f"{name}\t{values[2]:.4f}\t\t{values[3]:.4f}")

# Plot training accuracy
for metric, histories, ylabel in [
    ("Accuracy", accuracy_history, "Training Accuracy"),
    ("Loss", loss_history, "Training Loss")
]:
    plt.figure(figsize=(10, 6))
    for name, values in histories.items():
        plt.plot(range(1, epochs + 1), values, label=name)
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title("MNIST - " + ylabel)
    plt.legend()
    plt.grid()
    plt.tight_layout()
    plt.show()

In [ ]:
# EXP 8: Optmization Algorithms
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

def train(name, lr=0.01, epochs=50):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(30,)),
        tf.keras.layers.Dense(10, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    opts = {
        "sgd": tf.keras.optimizers.SGD(learning_rate=lr),
        "momentum": tf.keras.optimizers.SGD(learning_rate=lr, momentum=0.9),
        "nag": tf.keras.optimizers.SGD(learning_rate=lr, momentum=0.9, nesterov=True),
        "adagrad": tf.keras.optimizers.Adagrad(learning_rate=lr),
        "rmsprop": tf.keras.optimizers.RMSprop(learning_rate=lr),
        "adam": tf.keras.optimizers.Adam(learning_rate=lr)
    }

    model.compile(optimizer=opts[name],
                  loss="binary_crossentropy", metrics=["accuracy"])
    history = model.fit(X_train, y_train, epochs=epochs,
                        batch_size=1, verbose=0)
    _, accuracy = model.evaluate(X_test, y_test, verbose=0)
    return history.history["loss"], accuracy

optimizers = ["sgd", "momentum", "nag", "adagrad", "rmsprop", "adam"]

plt.figure(figsize=(8, 5))
for name in optimizers:
    loss, acc = train(name)
    plt.plot(loss, label=f"{name.upper()} ({acc:.4f})")

plt.title("Optimizer Comparison")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# EXP 9(A)
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

X, y = make_moons(n_samples=100, noise=0.30, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

def create_model(method="Baseline"):
    reg = (tf.keras.regularizers.L1(0.001) if method == "L1"
           else tf.keras.regularizers.L2(0.03) if method == "L2"
           else None)
    drop = 0.30 if method == "Dropout" else 0.0

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(2,)),
        tf.keras.layers.Dense(64, activation="relu", kernel_regularizer=reg),
        tf.keras.layers.Dropout(drop),
        tf.keras.layers.Dense(64, activation="relu", kernel_regularizer=reg),
        tf.keras.layers.Dropout(drop),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

rng = np.random.default_rng(SEED)
X_aug = np.vstack([X_train, X_train + rng.normal(0, 0.10, X_train.shape)])
y_aug = np.concatenate([y_train, y_train])

methods = ["Baseline", "L1", "L2", "Dropout",
           "Data Augmentation", "Early Stopping"]
colors = dict(zip(methods, ["black", "red", "blue",
                            "green", "purple", "orange"]))
models, results = {}, []

for method in methods:
    tf.keras.utils.set_random_seed(SEED)
    arch = method if method in ["Baseline", "L1", "L2", "Dropout"] else "Baseline"
    model = create_model(arch)
    xf, yf, args = X_train, y_train, {}

    if method == "Data Augmentation":
        xf, yf = X_aug, y_aug
    elif method == "Early Stopping":
        args = {"validation_split": 0.20,
                "callbacks": [tf.keras.callbacks.EarlyStopping(
                    monitor="val_loss", patience=30, restore_best_weights=True)]}

    hist = model.fit(xf, yf, epochs=1000, batch_size=4,
                     verbose=0, **args)
    tr_loss, tr_acc = model.evaluate(X_train, y_train, verbose=0)
    te_loss, te_acc = model.evaluate(X_test, y_test, verbose=0)

    models[method] = model
    results.append((method, tr_acc, te_acc, tr_loss, te_loss,
                    len(hist.history["loss"])))

print(f"{'Method':<20}{'Train Acc':>11}{'Test Acc':>11}"
      f"{'Train Loss':>12}{'Test Loss':>11}{'Epochs':>9}")
for r in results:
    print(f"{r[0]:<20}{r[1]:>11.4f}{r[2]:>11.4f}"
          f"{r[3]:>12.4f}{r[4]:>11.4f}{r[5]:>9}")

X_all = np.vstack([X_train, X_test])
x1 = np.linspace(X_all[:, 0].min()-1, X_all[:, 0].max()+1, 300)
x2 = np.linspace(X_all[:, 1].min()-1, X_all[:, 1].max()+1, 300)
xx, yy = np.meshgrid(x1, x2)
grid = np.c_[xx.ravel(), yy.ravel()]

def boundary(model):
    return model.predict(grid, verbose=0).reshape(xx.shape)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for row, (data, labels, title) in enumerate([
    (X_train, y_train, "Training"),
    (X_test, y_test, "Testing")
]):
    for col, selected in enumerate([
        ["Baseline"], methods[1:]
    ]):
        ax = axes[row, col]
        ax.scatter(data[labels == 0, 0], data[labels == 0, 1],
                   marker="o", facecolors="none", edgecolors="black",
                   label="Class 0")
        ax.scatter(data[labels == 1, 0], data[labels == 1, 1],
                   marker="^", color="black", label="Class 1")

        for method in selected:
            z = boundary(models[method])
            ax.contour(xx, yy, z, levels=[0.5],
                       colors=[colors[method]], linewidths=2)
            ax.plot([], [], color=colors[method], label=method)

        ax.set_title(f"{title} - {'Baseline' if col == 0 else 'Regularization'}")
        ax.set_xlabel("Feature 1")
        ax.set_ylabel("Feature 2")
        ax.legend(fontsize=8)

fig.suptitle("Baseline vs Regularization Techniques", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# 9(B)
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from sklearn.model_selection import train_test_split

SEED, EPOCHS, BATCH_SIZE = 42, 20, 32
NOISE_LEVEL = 0.10
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

(X1, y1), (X2, y2) = mnist.load_data()
X = np.concatenate([X1, X2]).astype("float32") / 255.0
y = np.concatenate([y1, y2])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

noise = np.random.normal(0, NOISE_LEVEL, X_train.shape).astype("float32")
X_train = np.clip(X_train + noise, 0, 1)[..., np.newaxis]
X_test = X_test[..., np.newaxis]

def create_model(method):
    reg = (tf.keras.regularizers.L1(0.001) if method == "L1"
           else tf.keras.regularizers.L2(0.001) if method == "L2"
           else None)

    layers = [tf.keras.layers.Input(shape=(28, 28, 1))]

    if method == "Data Augmentation":
        layers += [
            tf.keras.layers.RandomRotation(0.1, seed=SEED),
            tf.keras.layers.RandomZoom(0.1, seed=SEED)
        ]

    layers += [
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation="relu",
                              kernel_regularizer=reg,
                              kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED))
    ]

    if method == "Dropout":
        layers.append(tf.keras.layers.Dropout(0.5))

    layers.append(tf.keras.layers.Dense(10, activation="softmax"))
    model = tf.keras.Sequential(layers)
    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

methods = ["Baseline", "L1", "L2", "Dropout",
           "Data Augmentation", "Early Stopping"]
results = []

for method in methods:
    tf.keras.utils.set_random_seed(SEED)
    model = create_model(method)
    callbacks = ([tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True)]
        if method == "Early Stopping" else [])

    history = model.fit(
        X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE,
        validation_split=0.2, callbacks=callbacks, verbose=0
    )

    _, train_acc = model.evaluate(X_train, y_train, verbose=0)
    _, test_acc = model.evaluate(X_test, y_test, verbose=0)
    results.append((method, train_acc * 100, test_acc * 100,
                    len(history.history["loss"])))
    del model
    tf.keras.backend.clear_session()

print(f"{'Method':<20}{'Train Accuracy':>16}"
      f"{'Test Accuracy':>16}{'Epochs':>9}")
print("-" * 61)
for method, tr, te, ep in results:
    print(f"{method:<20}{tr:>15.2f}%{te:>15.2f}%{ep:>9}")

methods = [r[0] for r in results]
accuracies = [r[2] for r in results]

plt.figure(figsize=(10, 5))
bars = plt.bar(methods, accuracies)
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.3, f"{acc:.2f}%",
             ha="center", fontsize=9)

plt.xlabel("Methods")
plt.ylabel("Test Accuracy (%)")
plt.title("MNIST: Baseline vs Regularization Techniques")
plt.ylim(0, 100)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# EXP 10: CNN
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

X, y = load_digits(return_X_y=True)
X = (X / 16).reshape(-1, 8, 8, 1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

model = tf.keras.Sequential([
    tf.keras.layers.Input((8, 8, 1)),
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(10, activation="softmax")
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True)

h = model.fit(X_train, y_train, validation_split=0.2, epochs=100,
              batch_size=32, callbacks=[stop], verbose=0)
print("Training Accuracy:", h.history["accuracy"][-1])

plt.plot(h.history["loss"], label="Training")
plt.plot(h.history["val_loss"], label="Validation")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend(); plt.grid(); plt.show()

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print("Test Accuracy:", acc)
pred = model.predict(X_test, verbose=0).argmax(axis=1)
print(classification_report(y_test, pred))

sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion Matrix"); plt.show()

wrong = np.where(y_test != pred)[0]
plt.figure(figsize=(10, 5))
for i, idx in enumerate(wrong[:10]):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[idx].reshape(8, 8), cmap="gray")
    plt.title(f"T:{y_test[idx]} P:{pred[idx]}")
    plt.axis("off")
plt.suptitle("Misclassified Digits")
plt.tight_layout(); plt.show()